# 02 - Limpeza dos casos

**Objetivo:** deixar os dados do InfoDengue prontos para análise e modelagem.

Fonte: `data/raw/infodengue_teresina.csv` (baixado no `01_coleta`). O arquivo bruto **não é alterado**: tudo o que for tratado aqui será salvo em `data/processed/`.

## 1. Converter a data e ordenar no tempo

A API devolve um CSV com as semanas da mais recente pra mais antiga, e a data (`data_iniSE`) vem como texto. Converti e ordenei antes de qualquer coisa: com a ordem invertida, o lag de “semana anterior” pegaria a semana seguinte, e o modelo ia treinar vendo o futuro.



In [7]:
import pandas as pd

# Lê o dado bruto 
casos = pd.read_csv("../data/raw/infodengue_teresina.csv")
casos.shape

(615, 30)

In [8]:
# data_iniSE é texto e a primeira linha é a semana mais recente
print(casos["data_iniSE"].dtype)
casos[["data_iniSE", "SE", "casos", "casos_est"]].head(3)

str


,data_iniSE,SE,casos,casos_est
0,2026-10-11,202641,14,48.0
1,2026-10-04,202640,17,42.0
2,2026-09-27,202639,20,39.0


In [9]:
# Converte o texto "2015-01-04" para formato data.
# format diz o padrão do texto: %Y = ano, %m = mês, %d = dia.
# Não esquecer: se alguma linha fugir desse padrão, o pandas dá erro.
casos["data_iniSE"] = pd.to_datetime(casos["data_iniSE"], format="%Y-%m-%d")
casos["data_iniSE"].dtype

dtype('<M8[us]')

In [10]:
# Ordena do mais antigo para o mais recente
casos = casos.sort_values("data_iniSE")

# Depois de ordenar, o índice continua com os números antigos (613, 612, ...).
# reset_index renumera de 0 em diante. drop=True joga fora o índice velho em vez de virar coluna.
casos = casos.reset_index(drop=True)

casos[["data_iniSE", "SE", "casos", "casos_est"]].head(3)

,data_iniSE,SE,casos,casos_est
0,2015-01-04,201501,50,50.0
1,2015-01-11,201502,34,34.0
2,2015-01-18,201503,51,51.0


In [11]:
# Verifica se as últimas linhas devem ser as semanas mais recentes
casos[["data_iniSE", "SE", "casos", "casos_est"]].tail(3)

,data_iniSE,SE,casos,casos_est
612,2026-09-27,202639,20,39.0
613,2026-10-04,202640,17,42.0
614,2026-10-11,202641,14,48.0


In [12]:
# Retorna True se cada data é maior ou igual à anterior
print("Ordem crescente:", casos["data_iniSE"].is_monotonic_increasing)
print("Datas repetidas:", casos["data_iniSE"].duplicated().sum())
print("Formato:", casos.shape)

Ordem crescente: True
Datas repetidas: 0
Formato: (615, 30)


### Conclusões do passo 1

- `data_iniSE` agora é do tipo data (`datetime64`).
- As linhas estão em ordem cronológica: de 2015-01-04 (linha 0) até 2026-10-11 (última linha).
- Nenhuma data repetida e o formato continua 615 × 30 (nenhuma linha perdida).